# GAN-LPN PRNG: generator pseudolosowy wspierany przez GenAI

## Czym jest GAN w tym kontekście

GAN (*Generative Adversarial Network*) składa się z dwóch modeli uczonych jednocześnie:

- **generator** tworzy próbki, które mają wyglądać jak dane z pewnego docelowego rozkładu,
- **dyskryminator** albo predyktor próbuje wykryć, że próbki pochodzą od generatora, albo przewidzieć część wyjścia generatora.

W klasycznych zastosowaniach generator uczy się robić obrazy, tekstury albo inne dane podobne do przykładów treningowych. W PRNG idea jest inna: generator ma tworzyć ciąg bitów, który wygląda losowo albo którego kolejnych fragmentów nie da się łatwo przewidzieć na podstawie poprzednich. To kuszące, bo przypomina grę kryptograficzną: generator próbuje oszukać przeciwnika, a przeciwnik stale się uczy.

Trzeba jednak pamiętać o ważnym ograniczeniu: dobry wynik przeciwko uczonemu dyskryminatorowi nie jest jeszcze dowodem bezpieczeństwa kryptograficznego. Może oznaczać, że model przeszedł zestaw testów statystycznych, ale niekoniecznie, że jest odporny na wyspecjalizowaną analizę.


## Wczesny GAN PRNG: De Bernardi, Khouzani, Malacaria

Praca z 2018 r. proponuje trenowanie sieci neuronowej tak, aby zachowywała się jak PRNG. Autorzy opisują generator jako deterministyczną sieć feed-forward, która przyjmuje seed oraz prosty stan/licznik, a następnie produkuje krótki fragment wyjścia. W ich przykładowej architekturze generator ma kilka w pełni połączonych warstw, używa aktywacji Leaky ReLU i na końcu mapuje wynik do zakresu wyjściowego przez funkcję typu `mod`.

Autorzy badali dwa warianty uczenia:

- **wariant dyskryminacyjny** - dyskryminator dostaje próbki z generatora oraz próbki z zewnętrznego źródła losowości i uczy się rozróżniać, które są które; generator uczy się imitować rozkład danych losowych,
- **wariant predykcyjny** - wyjście generatora jest dzielone na część jawną i ukrytą; predyktor próbuje przewidzieć ukrytą część na podstawie jawnej, a generator uczy się utrudniać taką predykcję.

Ten drugi wariant nie polega tylko na kopiowaniu referencyjnego rozkładu. Generator ma produkować wartości trudne do przewidzenia dla przeciwnika uczącego się razem z nim. To przypomina uproszczoną wersję intuicji stojącej za testem następnego bitu: jeśli znając wcześniejsze bity nie umiemy przewidzieć kolejnego, ciąg zachowuje się bardziej losowo.

Wyniki były obiecujące statystycznie: autorzy raportowali około 99% zaliczonych instancji testów NIST i około 98% zaliczonych testów ogółem w najlepszym wariancie. Jednocześnie sami ograniczyli zakres pracy głównie do cech statystycznych wyjścia i nie przedstawili pełnej kryptanalizy. To ważne, bo PRNG może wyglądać losowo w testach, ale nadal mieć strukturę umożliwiającą atak.


## Od GAN PRNG do GAN-LPN PRNG

GAN PRNG z 2018 r. można traktować jako próbę odpowiedzi na pytanie: *czy sieć neuronowa może nauczyć się funkcji generatora pseudolosowego?* Odpowiedź eksperymentalna była częściowo pozytywna, ale nadal opierała się głównie na testach statystycznych.

GAN-LPN PRNG z nowszej pracy odpowiada na trochę inne pytanie: *czy model generatywny może być użyty jako składnik generatora, którego bezpieczeństwo opieramy na znanym trudnym problemie?* Dlatego GAN nie generuje tutaj bezpośrednio końcowego strumienia jako samodzielny PRNG. Jego zadaniem jest tworzenie kontrolowanego szumu Bernoulliego, który następnie trafia do rdzenia LPN.

To jest kluczowa różnica:

- w **GAN PRNG** bezpieczeństwo praktycznie zależy od tego, czy wytrenowana sieć produkuje ciągi trudne do odróżnienia/przewidzenia,
- w **GAN-LPN PRNG** sieć jest samplerem szumu, a główny argument bezpieczeństwa ma opierać się na trudności problemu LPN oraz na kontrolowaniu błędów samplera.


## Główna idea LPN GAN

Autorzy próbują połączyć dwie rzeczy:

- **GAN / WGAN-GP**: model uczy się produkować wektor szumu podobny do próbek z rozkładu Bernoulliego o zadanym prawdopodobieństwie jedynki, w pracy około $ p = 0.2 $.
- **LPN, czyli Learning Parity with Noise**: problem kryptograficzny, w którym obserwujemy liniowe równania nad bitami, ale część wyników jest zakłócona szumem. Odzyskanie sekretu jest trudne, gdy parametry i szum są dobrane poprawnie.

Sama sieć neuronowa nie jest tutaj traktowana jako pełna podstawa bezpieczeństwa. Jej rola jest bardziej ograniczona: ma dostarczać szum o kontrolowanych własnościach. Bezpieczeństwo ma wynikać z rdzenia LPN, a niedoskonałości samplera są w pracy ujmowane jako dodatkowe składniki pogarszające oszacowanie bezpieczeństwa.


## Składnik 1: BNES, czyli sampler szumu Bernoulliego

BNES (*Bernoulli Noise Expansion Sampler*) dostaje krótki seed i zwraca dłuższy wektor bitów. W instancji z pracy seed ma długość `l = 128`, a wektor szumu ma długość `n = 1024`.

Najprostsza intuicja: model ma działać jak deterministyczna maszyna, która z krótkiego wejścia tworzy 1024 bity wyglądające jak niezależne próbki Bernoulliego z prawdopodobieństwem jedynki bliskim `0.2`. Żeby to uzyskać, autorzy trenują generator i dyskryminator w wariancie WGAN-GP. Generator jest binarizowany na wyjściu przez BinarySTE: w przód daje bity `0/1`, a podczas uczenia pozwala przepuścić przybliżony gradient.

Dyskryminator nie sprawdza tylko, czy próbka wygląda losowo. Ma też głowy oceniające średnią i zależności między bitami, bo zbyt duże korelacje osłabiłyby założenie LPN.


## Składnik 2: czym jest LPN i rdzeń LPN generatora

LPN (*Learning Parity with Noise*) można intuicyjnie opisać jako rozwiązywanie układu równań nad bitami, ale z błędami. Bez szumu sprawa jest prosta: jeśli mamy wystarczająco dużo równań liniowych nad `0/1`, można odzyskać tajny wektor metodami algebry liniowej. Szum psuje część odpowiedzi, więc atakujący nie wie, które równania są poprawne, a które zostały odwrócone.

Typowa próbka LPN wygląda tak:

$$ b = A s \oplus e $$

gdzie $A$ jest znaną macierzą bitową, $s$ jest tajnym wektorem, a $e$ jest wektorem szumu. Symbol $\oplus$ oznacza XOR. Trudność polega na odzyskaniu $s$ albo odróżnieniu takich próbek od losowych, gdy szum jest dobrany sensownie.

Generator utrzymuje stan złożony z dwóch części:

- $ v_t $: tajny stan liniowy, w pracy 320 bitów,
- $ r_t $: seed dla BNES, w pracy 128 bitów.

W każdej rundzie BNES tworzy wektor szumu $ e_t $. Następnie generator liczy liniową transformację stanu i miesza ją z tym szumem operacją XOR:


$$ c_t = M^T v_t \oplus e_t $$

Macierz $M$ to stała binarna macierz klucza, czyli ustalony „przepis mieszania” bitów stanu. Jej elementy są bitami `0/1`, a mnożenie przez $M^T$ wybiera i miesza odpowiednie kombinacje bitów z $v_t$. Dopiero potem wynik jest zakłócany szumem $e_t$ z BNES. Dzięki temu generator ma strukturę LPN: najpierw liniowe mieszanie tajnego stanu, a potem dodanie kontrolowanego szumu.


Wektor $ c_t $ jest potem dzielony na trzy części: nowy $ v $, nowy $r$ oraz bity wyjściowe $ z_t $. Dla parametrów z tabeli w pracy ($ n = 1024 $, $ m = 320 $, $ l = 128 $) jedna runda daje $ mu = n - m - l = 576 $ bitów wyjściowych.

W skrócie - poprzedni stan wyznacza część strukturalną, BNES dodaje kontrolowany szum, a wynik jednocześnie aktualizuje stan i produkuje porcję bitów.


## Algorytm w skrócie

Poniżej sformułowano pseudokod opisujący najwazniejsze operacje.

```text
Dane: macierz M, seed = v_0 || r_0

dla kolejnych rund t:
    e_t = BNES(r_t)                 # wytrenowany GAN tworzy szum Bernoulliego
    c_t = M^T v_t XOR e_t           # rdzeń LPN

    v_{t+1} = pierwsze m bitów c_t
    r_{t+1} = kolejne l bitów c_t
    z_t     = pozostałe bity c_t

    zwróć z_t
```

## Notatki do odtworzenia części implementacyjnej z repozytorium autorów

Sprawdzona wersja repozytorium nie zawiera gotowego pliku wag `gan_model_full_output_run_0_1024.pdparams`, którego adapter używa do generowania bitów.

Repozytorium autorów zawiera instrukcję treningu, więc implementacja oraz walidacja wyników są możliwe. Minimalna ścieżka wygląda tak:

Repozytorium znajduje się tu:
```bash
git clone https://gitee.com/guangandy/ganlpnprng.git
```
Następnie z README autorów wynika, że potrzebne są:

```bash
pip install paddlepaddle numpy scipy matplotlib randomgen cryptography psutil tabulate jupyter
```

Uruchoimienie treningu:

```bash
python traingan.py
```

Po treningu kod zapisuje pliki `.pdparams` z wagami generatora i dyskryminatora. Dopiero wtedy można uruchomić adapter PRNG z repozytorium autorów albo przygotować lokalny wrapper zgodny z interfejsem używanym w tych notatnikach.

W praktyce trzeba jeszcze zwrócić uwagę na nazwy plików wag, bo w sprawdzonej wersji kodu występują różne konwencje nazewnicze, np. `1024gan_model_full_output_run_0.pdparams` w skrypcie treningowym oraz `gan_model_full_output_run_0_1024.pdparams` w adapterze/testach. To nie zmienia idei generatora, ale wymaga uporządkowania przed automatyczną walidacją.


## Artefakty wynikowe z repozytorium autorów

W repozytorium autorów znajduje się też katalog `results/`. Nie zawiera on wag BNES potrzebnych do uruchomienia GAN-LPN, ale zawiera gotowe artefakty eksperymentalne, o których warto wspomnieć:

- `gan_lpn_prng_hd.png` - wykres testu wrażliwości na seed dla proponowanego GAN-LPN PRNG,
- `aes_cbc_hd.png`, `bbs_prng_hd.png`, `chacha_prng_hd.png`, `hc128_prng_hd.png`, `lpn_prng_hd.png`, `mt_prng_hd.png`, `gan_prng_hd.png` - analogiczne wykresy dla generatorów porównawczych,
- `prng_performance_results_20260120_121805.json` - zapis benchmarków wydajności, latencji i użycia CPU/energii dla kilku generatorów.

Najbardziej użyteczny dla tego notatnika jest wykres odległości Hamminga dla GAN-LPN. Autorzy zmieniają kolejne pozycje bitowe seeda i mierzą, jaka część bitów w wyniku ulega zmianie. Dla dobrego efektu lawinowego oczekujemy wartości bliskiej `0.5`, czyli zmiany około połowy bitów wyjściowych.

> **Odległość Hamminga:** 
>
> mówi, ile elementów różni się między dwoma ciągami tej samej długości.
> Najprościej:
> ```
> 101100
> 100110
> ```
> Te ciągi różnią się w 2 miejscach, więc ich odległość Hamminga to 2. 
>
> W generatorach losowych używa się jej do sprawdzenia, czy mała zmiana seeda mocno zmienia wynik


<img src="P10-assets/gan_lpn_prng_hd.png" width="720">

Na wykresie średnia wynosi około `0.50007`, bardzo blisko wartości idealnej `0.5`. To wspiera opis jakości efektu lawinowego, ale nie zastępuje lokalnej walidacji w tym projekcie.

Plik benchmarkowy JSON podaje m.in. wpisy dla `GAN LPN PRNG`, `GAN PRNG`, `AES_CTR_DRBG`, `ChaCha20`, `HC128`, `BBS`, `MT19937` i `LPN PRNG`. Dla GAN-LPN widoczne są wyniki zależne od trybu i batch size; np. wariant GPU z batch size `100` ma throughput około `4.32 MB/s`, co odpowiada wartości raportowanej w artykule. Te liczby traktuję jako wyniki autorów, nie jako wynik uruchomienia w lokalnym środowisku.


## Źródła

> Marcello De Bernardi, MHR Khouzani, Pasquale Malacaria, *Pseudo-Random Number Generation using Generative Adversarial Networks*, 2018 - jedna z wczesnych prób użycia GAN bezpośrednio do trenowania PRNG,

> Xuguang Wu i in., *Generative artificial intelligence-driven secure pseudorandom number generator*, 2026 - konstrukcja GAN-LPN, w której GAN nie jest samodzielną podstawą bezpieczeństwa, lecz generuje szum dla rdzenia opartego o problem LPN.

> Repozytorium autorów Xuguang Wu i in. : `https://gitee.com/guangandy/ganlpnprng`
